# Habitat RGB-D probe — map 00805

Attach the private `rebuild-rgbd-00805-input` Dataset built by `prepare_input.py` (the ZIP can be uploaded as one file). This notebook tests headless Habitat rendering and sensor capture; it does not start NeuSymNav M1–M5.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib, json, os, shutil, subprocess, sys, time
from IPython.display import Image, display

manifests = [p for p in Path('/kaggle/input').glob('**/manifest.json')
             if json.loads(p.read_text()).get('bundle') == 'habitat-rgbd-probe-v1']
if not manifests:
    archives = list(Path('/kaggle/input').glob('**/*.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Attach the input bundle folder or exactly one bundle zip; found {len(archives)} zips')
    extracted = Path('/kaggle/working/rebuild-rgbd-00805-input')
    extracted.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as zf:
        for name in zf.namelist():
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise ValueError(f'Unsafe path in input zip: {name}')
        zf.extractall(extracted)
    manifests = [p for p in extracted.rglob('manifest.json')
                 if json.loads(p.read_text()).get('bundle') == 'habitat-rgbd-probe-v1']
if len(manifests) != 1:
    raise RuntimeError(f'Expected one probe manifest, found {len(manifests)}')
bundle = manifests[0].parent
manifest = json.loads(manifests[0].read_text())
print('bundle:', bundle)
print('map:', manifest['map'], 'Rebuild pins:', manifest['rebuild_pins_checked_before_staging'])
print('Python:', sys.version.split()[0], 'kernel:', sys.executable)
print('working free GiB:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10).stdout[:1500] or 'No NVIDIA GPU visible')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('No nvidia-smi; Habitat probe does not require a CUDA GPU.')

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
scene = next((bundle / 'scene').rglob('*.basis.glb'))
navmesh = scene.with_suffix('.navmesh')
scene_dataset = bundle / 'scene/hm3d_minival_basis.scene_dataset_config.json'
source_root = bundle / 'Rebuild/src/standalone_sg'
source_files = sorted(source_root.rglob('*.py'))
tree = hashlib.sha256()
for path in source_files:
    tree.update(path.relative_to(source_root).as_posix().encode())
    tree.update(b'\0')
    tree.update(bytes.fromhex(sha256(path)))
pins = manifest['rebuild_pins_checked_before_staging']
assert sha256(scene) == manifest['scene_sha256']
assert sha256(navmesh) == manifest['navmesh_sha256']
assert sha256(scene_dataset) == manifest['scene_dataset_sha256']
assert sha256(bundle / 'Rebuild/build/native-m3/occupancy/standalone_sg_octomap_fixture_runner') == pins['m23']
assert sha256(bundle / 'Rebuild/build/native-m4-dude/occusg_dude_worker') == pins['m4']
assert len(source_files) == pins['source_file_count'] == 50 and tree.hexdigest() == pins['source']
print('Bundle hashes and pinned Rebuild source closure verified.')

In [ ]:
# Use Habitat-Sim 0.3.3's Python 3.9 headless EGL package.
habitat_python = None
try:
    import habitat_sim
    if habitat_sim.__version__ == '0.3.3':
        habitat_python = Path(sys.executable)
        print('Using Habitat already installed in the notebook kernel.')
except (ImportError, OSError):
    pass
if habitat_python is None:
    manager = shutil.which('conda') or shutil.which('micromamba')
    if manager is None:
        # Kaggle images may not include Conda; bootstrap micromamba in working storage.
        import io, tarfile, urllib.request
        archive = urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest', timeout=90).read()
        with tarfile.open(fileobj=io.BytesIO(archive), mode='r:bz2') as tar:
            member = tar.getmember('bin/micromamba')
            target = Path('/kaggle/working/bin/micromamba')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(tar.extractfile(member).read())
            target.chmod(0o755)
        manager = str(target)
    env_prefix = Path('/kaggle/working/envs/habitat-039')
    habitat_python = env_prefix / 'bin/python'
    if not habitat_python.exists():
        cmd = [manager, 'create', '-y', '-p', str(env_prefix), 'python=3.9',
               'habitat-sim=0.3.3', 'headless', '-c', 'conda-forge', '-c', 'aihabitat']
        print('Installing pinned Habitat-Sim 0.3.3 headless environment; Internet must be enabled.')
        subprocess.run(cmd, check=True, env=dict(os.environ, MAMBA_ROOT_PREFIX='/kaggle/working/mamba-root'))
probe = subprocess.run([str(habitat_python), '-c',
    'import habitat_sim; print(habitat_sim.__version__, habitat_sim.cuda_enabled)'],
    check=True, capture_output=True, text=True)
print('Habitat import:', probe.stdout.strip())

In [ ]:
profile = bundle / 'profile.json'
worker = bundle / 'project/src/neusymnav/sensors/habitat_worker.py'
output = Path('/kaggle/working/rebuild-rgbd-00805-probe')
env = dict(os.environ, EGL_PLATFORM='surfaceless', PYTHONDONTWRITEBYTECODE='1')
start = time.monotonic()
subprocess.run([str(habitat_python), str(bundle / 'probe/habitat_probe.py'),
    '--profile', str(profile), '--scene', str(scene), '--scene-dataset', str(scene_dataset),
    '--worker', str(worker), '--output', str(output)], check=True, env=env)
print('notebook elapsed seconds:', round(time.monotonic() - start, 2))

In [ ]:
report = json.loads((output / 'habitat_probe.json').read_text())
assert report['status'] == 'passed'
print(json.dumps(report, indent=2))
display(Image(filename=str(output / 'rgb.png')))
display(Image(filename=str(output / 'depth.png')))
print('Saved outputs:', *[str(p) for p in sorted(output.iterdir())], sep='\n - ')